<a href="https://colab.research.google.com/github/nadilHesara/langid_experiments/blob/main/LangID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Data Loading

###sinhala paali parallel corpus data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

path = "/content/drive/MyDrive/SSLI/raw_data/sinhala_paali_parallel_corpus/output.tsv"

df = pd.read_csv(path, sep="\t")
print(df.shape)
print(df.columns.tolist())
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(28395, 2)
['pali_text', 'sinhala_text']


,pali_text,sinhala_text
0,එවං මෙ සුතං - එකං සමයං භගවා අන්තරා ච රාජගහං අන...,මා විසින් මෙසේ අසනලදී. (නොහොත් මගේ ඇසීම මෙලෙසය...
1,අථ ඛො භගවා අම්බලට්ඨිකායං රාජාගාරකෙ එකරත්තිවාසං...,එවිට භාග්‍යවතුන් වහන්සේ අම්බලට්ඨිකා උයනෙහි රජු...
2,අථ ඛො සම්බහුලානං භික්ඛූනං රත්තියා පච්චූසසමයං ප...,එකල රෑ පාන්දරින් මණ්ඩලමාල නම් රැස්වීම් ශාලාවෙහ...
3,අථ ඛො භගවා තෙසං භික්ඛූනං ඉමං සඞ්ඛියධම්මං විදිත...,එකල්හි භාග්‍යවතුන් වහන්සේ භික්ෂූන්ගේ මේ කථාව ද...
4,"‘‘මමං වා, භික්ඛවෙ, පරෙ අවණ්ණං භාසෙය්‍යුං, ධම්ම...","“මහණෙනි, අනුන් මාගේ හෝ නුගුණ කියද්ද, ධර්මය ගැන..."


###doc data

####paali

In [ ]:
import os

folder = "/content/drive/.shortcut-targets-by-id/1gPCbEcE9ztxyuWFqpVtSSNpel4wklH_g/SSLI/raw_data/doc/paali"

txt_files = [f for f in os.listdir(folder) if f.endswith(".txt")]
print(f"Found {len(txt_files)} files:")
for f in txt_files:
    print(f)

Found 15 files:
බුද්ධ සික්ඛා.txt
අධිමාස දීපනය.txt
මහාවංස ටීකා.txt
සුමංගල විලාසිනි නාමා දීඝනිකාය අට්ඨකථා.txt
සාදුපත්ථපුරණී.txt
සාදුජනපපසාදිනී.txt
සමෙමාහනාසිනී.txt
මහාරූපසිද්ධි.txt
මොග්ගලලායනව්_යාකරණං.txt
මාහාවංසො (තුතියෝ භාගෝ).txt
අධිකරණ විභාග සංඝෝ.txt
ප්_රපංච සුදනීනාම මජ්ජිම නිකාය අට්ඨකථා හෙවත් මැදුම් සඟි අටුවාව.txt
දීපවංසෝ.txt
ධම්මපදට්ඨ කථා - දුතියො භාගො.txt
ජාතකට්ඨකථා.txt


In [ ]:
import re, pandas as pd

label_map = {
    "අධිමාස": "pali", "කුදුසික": "pali", "ටීකා": "pali",
    "සුමංගල": "pali", "සාදුපත්ථ": "pali", "සාදුජනප": "pali",
    "සමෙමාහන": "pali", "බුද්ධ සික්ඛා": "pali", "මහාරූපසිද්ධි": "pali",
    "ග්ගලලායන": "pali", "අධිකරණ": "pali", "මාහාවංසො": "pali",
    "ප්_රපංච": "pali", "ප්රපංච": "pali",
    "පබ්බතූපම": "pali_then_sinhala",
    "ධම්මපදට්ඨ": "pali", "දීපවංසෝ": "pali", "ජාතකට්ඨකථා": "pali",
    "කවච": "sanskrit_from_line40",
    "චරියා": "EXCLUDE_COPYRIGHT",
}

import unicodedata

def normalize(s):
    return unicodedata.normalize("NFC", s)

def find_label(filename):
    filename_norm = normalize(filename)
    for key, val in label_map.items():
        if normalize(key) in filename_norm:
            return val
    return "UNKNOWN"

rows = []
skipped = []

for fname in txt_files:
    label = find_label(fname)
    book_title = fname.replace(".txt", "")

    if label in ("EXCLUDE_COPYRIGHT", "UNKNOWN"):
        skipped.append((fname, label))
        continue

    with open(os.path.join(folder, fname), encoding="utf-8") as f:
        text = f.read()

    lines = [l.strip() for l in text.split("\n")]
    lines = [l for l in lines if len(l) > 5 and not re.fullmatch(r"[\d\.\s]+", l)]

    if label == "pali_then_sinhala":
        skipped.append((fname, "needs manual pali/sinhala split"))
        continue

    if label == "sanskrit_from_line40":
        lines = lines[39:]
        label = "sanskrit"

    for sent in lines:
        rows.append({
            "text": sent,
            "label": label,
            "source": "SiDiaC-v2",
            "subcorpus": book_title,
            "group_id": f"sidiac2_{book_title}"
        })

sidiac_df = pd.DataFrame(rows)
print(sidiac_df.shape)
print(sidiac_df["label"].value_counts())
print("\nSkipped/flagged:")
for s in skipped:
    print(s)

(2929, 5)
label
pali    2929
Name: count, dtype: int64

Skipped/flagged:


####sanskrit

In [ ]:
!find /content/drive -type d -iname "*sasnkrit*" 2>/dev/null

/content/drive/.shortcut-targets-by-id/1gPCbEcE9ztxyuWFqpVtSSNpel4wklH_g/SSLI/raw_data/doc/sasnkrit


In [ ]:
import os

sanskrit_folder = "/content/drive/.shortcut-targets-by-id/1gPCbEcE9ztxyuWFqpVtSSNpel4wklH_g/SSLI/raw_data/doc/sasnkrit"
print(os.listdir(sanskrit_folder))

['කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහයන් පිළිබඳ කවචයන් සහ දේවි කවචය ආදි කවචයන්.txt']


In [ ]:
import re

fname = os.listdir(sanskrit_folder)[0]  # only one file in there
with open(os.path.join(sanskrit_folder, fname), encoding="utf-8") as f:
    text = f.read()

lines = [l.strip() for l in text.split("\n")]
lines = lines[39:]  # skip first 39 lines (0-indexed), keep line 40 onwards
lines = [l for l in lines if len(l) > 5 and not re.fullmatch(r"[\d\.\s]+", l)]

sanskrit_rows = pd.DataFrame([{
    "text": sent,
    "label": "sanskrit",
    "source": "SiDiaC-v2",
    "subcorpus": fname.replace(".txt", ""),
    "group_id": f"sidiac2_{fname.replace('.txt', '')}"
} for sent in lines])

print(sanskrit_rows.shape)
sanskrit_rows.head()

(101, 5)


,text,label,source,subcorpus,group_id
0,නව ග්‍රහාණං කවචං වක්ෂෙහං ශ්‍රැණු සුව්‍රත,sanskrit,SiDiaC-v2,කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහයන් පිළි...,sidiac2_කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහ...
1,"සර්ව සම්පත් ප්‍රදංචෛව සර්වරොග හර පරම්,",sanskrit,SiDiaC-v2,කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහයන් පිළි...,sidiac2_කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහ...
2,සර්ව ශත්‍රැක්ෂය චෛව භුක්ති මුක්ති ප්‍රදංශුභං,sanskrit,SiDiaC-v2,කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහයන් පිළි...,sidiac2_කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහ...
3,ඒකාන්ත සුද්ධ දෙශෙතු සුඛාසීන උදං මුඛ්‍ය,sanskrit,SiDiaC-v2,කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහයන් පිළි...,sidiac2_කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහ...
4,රීෂ්ටාදීනාසතංකෘත්වා තත්තන්ධ්‍යා නෛශවතොෂයෙත්,sanskrit,SiDiaC-v2,කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහයන් පිළි...,sidiac2_කවච සංග්_රහය හෙවත් ආදිත්_යාදි නව ග්_රහ...


##merging into a full dataset

In [ ]:
lines = [l for l in lines if len(l) > 5 and not re.fullmatch(r"[\d\.\s]+", l)]

sanskrit_rows = pd.DataFrame([{
    "text": sent,
    "label": "sanskrit",
    "source": "SiDiaC-v2",
    "subcorpus": fname.replace(".txt", ""),
    "group_id": f"sidiac2_{fname.replace('.txt', '')}"
} for sent in lines])

print("Sanskrit rows:", sanskrit_rows.shape)

sidiac_df = pd.concat([sidiac_df, sanskrit_rows], ignore_index=True)
print(sidiac_df["label"].value_counts())

Sanskrit rows: (101, 5)
label
pali        2929
sanskrit     101
Name: count, dtype: int64


In [ ]:
pali_rows = df[["pali_text"]].rename(columns={"pali_text": "text"})
pali_rows["label"] = "pali"

sinhala_rows = df[["sinhala_text"]].rename(columns={"sinhala_text": "text"})
sinhala_rows["label"] = "sinhala"

parallel_long = pd.concat([pali_rows, sinhala_rows], ignore_index=True)
parallel_long["source"] = "pali-sinhala-parallel"
parallel_long["subcorpus"] = "parallel_corpus"
parallel_long["group_id"] = "parallel_" + parallel_long.index.astype(str)

print(parallel_long.shape)
print(parallel_long["label"].value_counts())

(56790, 5)
label
pali       28395
sinhala    28395
Name: count, dtype: int64


In [ ]:
unified = pd.concat([sidiac_df, parallel_long], ignore_index=True)

before = len(unified)
unified = unified.drop_duplicates(subset="text").reset_index(drop=True)
print(f"Removed {before - len(unified)} duplicate sentences")

unified["id"] = range(len(unified))
unified = unified[["id", "text", "label", "source", "subcorpus", "group_id"]]

print(unified.shape)
print(unified["label"].value_counts())
unified.head()

Removed 3866 duplicate sentences
(55954, 6)
label
pali        29317
sinhala     26536
sanskrit      101
Name: count, dtype: int64


,id,text,label,source,subcorpus,group_id
0,0,බුද්ද සික්ඛා,pali,SiDiaC-v2,බුද්ධ සික්ඛා,sidiac2_බුද්ධ සික්ඛා
1,1,නමො තස්ස භගවතො අරහතො සම්මා සම්බුද්ධස්ස;,pali,SiDiaC-v2,බුද්ධ සික්ඛා,sidiac2_බුද්ධ සික්ඛා
2,2,1) ආදිතො උපසම්පන් සික්ඛිතබ්බං සමාතිකං,pali,SiDiaC-v2,බුද්ධ සික්ඛා,sidiac2_බුද්ධ සික්ඛා
3,3,බුද්දසික්ඛං පවක්ඛාමි වන්දිත්වා රතනත්තයං --,pali,SiDiaC-v2,බුද්ධ සික්ඛා,sidiac2_බුද්ධ සික්ඛා
4,4,තස්ස භගවතො අරහතො සම්මා සම්බුද්ධස්ස;,pali,SiDiaC-v2,බුද්ධ සික්ඛා,sidiac2_බුද්ධ සික්ඛා


In [ ]:
print(unified.groupby("label")["group_id"].nunique())

label
pali        26435
sanskrit        1
sinhala     26536
Name: group_id, dtype: int64


In [ ]:
def group_stratified_split(data, group_col, label_col, source_col="source", seed=42, min_groups_for_group_split=3):
    splits = {"train": [], "val": [], "test": []}

    for (label, source), sub in data.groupby([label_col, source_col]):
        n_groups = sub[group_col].nunique()

        if n_groups < min_groups_for_group_split:
            from sklearn.model_selection import train_test_split
            train, temp = train_test_split(sub, test_size=0.2, random_state=seed)
            val, test = train_test_split(temp, test_size=0.5, random_state=seed)
        else:
            gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
            train_idx, temp_idx = next(gss1.split(sub, groups=sub[group_col]))
            train = sub.iloc[train_idx]
            temp = sub.iloc[temp_idx]

            gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
            val_idx, test_idx = next(gss2.split(temp, groups=temp[group_col]))
            val = temp.iloc[val_idx]
            test = temp.iloc[test_idx]

        splits["train"].append(train)
        splits["val"].append(val)
        splits["test"].append(test)

    return {k: pd.concat(v, ignore_index=True) for k, v in splits.items()}

splits = group_stratified_split(unified, group_col="group_id", label_col="label", source_col="source")

for name, part in splits.items():
    print(f"\n--- {name} ---")
    print(part.groupby(["label", "source"]).size())


--- train ---
label     source               
pali      SiDiaC-v2                 2354
          pali-sinhala-parallel    21136
sanskrit  SiDiaC-v2                   80
sinhala   pali-sinhala-parallel    21228
dtype: int64

--- val ---
label     source               
pali      SiDiaC-v2                 158
          pali-sinhala-parallel    2642
sanskrit  SiDiaC-v2                  10
sinhala   pali-sinhala-parallel    2654
dtype: int64

--- test ---
label     source               
pali      SiDiaC-v2                 385
          pali-sinhala-parallel    2642
sanskrit  SiDiaC-v2                  11
sinhala   pali-sinhala-parallel    2654
dtype: int64


###Building Simple Classifier

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

train = splits["train"]
val = splits["val"]
test = splits["test"]

# character n-grams (2-4 chars) — works well for closely related scripts
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), max_features=20000)

X_train = vectorizer.fit_transform(train["text"])
X_val = vectorizer.transform(val["text"])
X_test = vectorizer.transform(test["text"])

y_train = train["label"]
y_val = val["label"]
y_test = test["label"]

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [ ]:
val_preds = clf.predict(X_val)
print(classification_report(y_val, val_preds))

              precision    recall  f1-score   support

        pali       1.00      1.00      1.00      2800
    sanskrit       1.00      0.90      0.95        10
     sinhala       1.00      1.00      1.00      2654

    accuracy                           1.00      5464
   macro avg       1.00      0.97      0.98      5464
weighted avg       1.00      1.00      1.00      5464



In [ ]:
# Check: does the model do well because of SOURCE, not LANGUAGE?
val_with_preds = val.copy()
val_with_preds["pred"] = val_preds
val_with_preds["correct"] = val_with_preds["pred"] == val_with_preds["label"]

print(val_with_preds.groupby(["label", "source"])["correct"].agg(["mean", "count"]))

                                    mean  count
label    source                                
pali     SiDiaC-v2              0.987342    158
         pali-sinhala-parallel  0.999621   2642
sanskrit SiDiaC-v2              0.900000     10
sinhala  pali-sinhala-parallel  1.000000   2654


In [ ]:
print(unified.groupby(["label", "source"])["group_id"].nunique())
print()
for name, part in splits.items():
    print(f"--- {name} ---")
    print(part.groupby(["label", "source"]).size())
    print()

label     source               
pali      SiDiaC-v2                   15
          pali-sinhala-parallel    26420
sanskrit  SiDiaC-v2                    1
sinhala   pali-sinhala-parallel    26536
Name: group_id, dtype: int64

--- train ---
label     source               
pali      SiDiaC-v2                 2354
          pali-sinhala-parallel    21136
sanskrit  SiDiaC-v2                   80
sinhala   pali-sinhala-parallel    21228
dtype: int64

--- val ---
label     source               
pali      SiDiaC-v2                 158
          pali-sinhala-parallel    2642
sanskrit  SiDiaC-v2                  10
sinhala   pali-sinhala-parallel    2654
dtype: int64

--- test ---
label     source               
pali      SiDiaC-v2                 385
          pali-sinhala-parallel    2642
sanskrit  SiDiaC-v2                  11
sinhala   pali-sinhala-parallel    2654
dtype: int64

